In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df=pd.read_csv(r"C:\Users\AJAY PATEL\Downloads\power+consumption+of+tetouan+city\Tetuan City power consumption.csv")
df.head()

,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,1/1/2017 0:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
1,1/1/2017 0:10,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2,1/1/2017 0:20,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
3,1/1/2017 0:30,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
4,1/1/2017 0:40,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964


In [6]:
df.shape

(52416, 9)

In [8]:
df.isnull().sum()

DateTime                     0
Temperature                  0
Humidity                     0
Wind Speed                   0
general diffuse flows        0
diffuse flows                0
Zone 1 Power Consumption     0
Zone 2  Power Consumption    0
Zone 3  Power Consumption    0
dtype: int64

In [10]:
df['DateTime'] = pd.to_datetime(df['DateTime'])

df['hour'] = df['DateTime'].dt.hour
df['day'] = df['DateTime'].dt.day
df['month'] = df['DateTime'].dt.month

df.drop(['DateTime'], axis=1, inplace=True)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52416 entries, 0 to 52415
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Temperature                52416 non-null  float64
 1   Humidity                   52416 non-null  float64
 2   Wind Speed                 52416 non-null  float64
 3   general diffuse flows      52416 non-null  float64
 4   diffuse flows              52416 non-null  float64
 5   Zone 1 Power Consumption   52416 non-null  float64
 6   Zone 2  Power Consumption  52416 non-null  float64
 7   Zone 3  Power Consumption  52416 non-null  float64
 8   hour                       52416 non-null  int32  
 9   day                        52416 non-null  int32  
 10  month                      52416 non-null  int32  
dtypes: float64(8), int32(3)
memory usage: 3.8 MB


In [14]:
X = df.drop("Zone 1 Power Consumption", axis=1)
y = df["Zone 1 Power Consumption"]

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
transformer = ColumnTransformer(transformers=[("t2",StandardScaler(),[0,1,2,3,4])])
X_train_trans = transformer.fit_transform(X_train)
X_test_trans = transformer.transform(X_test)

In [26]:
X_train_trans = pd.DataFrame(X_train_trans)
X_train_trans.head()

,0,1,2,3,4
0,1.959653,-1.045386,1.263966,0.938819,0.283407
1,0.095954,1.439384,-0.702933,-0.693397,-0.604490
2,1.925140,-2.208210,1.259707,-0.426565,-0.021765
3,-1.117176,1.097945,-0.801299,-0.693465,-0.604224
4,-0.123203,-0.376038,-0.795764,-0.414853,0.065197


In [24]:
X_test_trans = pd.DataFrame(X_test_trans)
X_test_trans.head()

,0,1,2,3,4
0,0.680949,-2.482005,-0.797467,1.631712,4.747448
1,-0.499394,-0.890128,-0.797467,-0.621437,-0.449810
2,-0.644348,-0.710390,-0.801299,0.804699,-0.278141
3,-1.103371,0.260455,-0.799170,-0.693439,-0.604345
4,-0.730631,0.795160,-0.799596,-0.693605,-0.603894


In [64]:
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

model.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                      random_state=42)

In [66]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [76]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [78]:
print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))
print("Test RMSE:", mean_squared_error(y_test, y_test_pred, squared=False))

Train R2: 0.9692327142115623
Test R2: 0.9625922657664248
Test RMSE: 1374.3655215957076


In [80]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

In [82]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [84]:
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2']
}

In [86]:
grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [5, 10, 15, 20, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 5, 10],
                         'min_samples_split': [2, 5, 10, 20]},
             scoring='r2')

In [88]:
print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 10}


In [90]:
best_model = grid.best_estimator_

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred))

Train R2: 0.99540300427584
Test R2: 0.9784228589482997
Test RMSE: 1043.8029017943775


In [104]:
param_dist = {
    'max_depth': [None] + list(np.arange(5, 30, 5)),
    'min_samples_split': np.arange(2, 21, 2),
    'min_samples_leaf': np.arange(1, 11, 1),
    'max_features': [None, 'sqrt', 'log2']
}

In [106]:
random_search = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=30,              
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'max_depth': [None, 5, 10, 15, 20, 25],
                                        'max_features': [None, 'sqrt', 'log2'],
                                        'min_samples_leaf': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
                                        'min_samples_split': array([ 2,  4,  6,  8, 10, 12, 14, 16, 18, 20])},
                   random_state=42, scoring='r2')

In [107]:
print("Best Parameters:", random_search.best_params_)

Best Parameters: {'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': None, 'max_depth': 20}


In [108]:
best_model = random_search.best_estimator_

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred))

Train R2: 0.9936822093190646
Test R2: 0.9782907445644523
Test RMSE: 1046.9935685294836
